# NPS-25-003

Template: Getting_started.ipynb from hepdata_lib

From hepdata_lib:
The following instructions and examples should get you started to get your analysis into [HEPData](https://hepdata.net) using `hepdata_lib`. Please also refer to the [documentation](http://hepdata-lib.readthedocs.io/). While you can also run `hepdata_lib` on your local computer, you can use the [binder](https://mybinder.org/) or [SWAN](http://swan.cern.ch/) services in the browser. Mind that SWAN is only available for people with a CERN account.

Also useful reference: https://github.com/jalimena/HepData_EXO-23-016/tree/main
See "main" function in createHepData_all.py

## General setup

To make sure things are working and `hepdata_lib` is available, run the following command:

In [25]:
import hepdata_lib
import numpy as np
from hepdata_lib import Submission, Table, Variable
from __future__ import print_function
print("hepdata_lib version", hepdata_lib.__version__)

hepdata_lib version 0.20.0


## Adding a table/figure

In HEPData, figures and table will both be `Table` objects. 

The first column is the mass of phi_2, the second of phi_1, and the third is the median upper limit.

Let's create the table/figure. First, we need to give it a name, which is usually just the identifier in the paper, i.e. "Figure _". The table also needs a description, which is usually the caption. You also need to describe the location, i.e. where to find it in the publication:

In [26]:
def make2DLimitTable(tableName, isBDT, fileName, imageName):

    table = Table(tableName)
    if isBDT:
        table.description = (
            "The 95% CL upper limits on the products $\\sigma B_\\mathrm{C}$ and"
            " $\\sigma B_\\mathrm{NC}$, obtained using the BDT-based event categorization,"
            " as a function of the scalar masses $m_{\\phi_1}$ and $m_{\\phi_2}$."
            " For the $(m_{\\phi_1}, m_{\\phi_2})$ mass hypotheses (15, 30), (20, 40),"
            " and (30, 60) GeV, only the non-cascade limits are shown. For all other mass"
            " hypotheses, either cascade or non-cascade limits are presented, depending on"
            " whether the cascade decay is kinematically allowed"
            " ($m_{\\phi_2} \\geq 2 m_{\\phi_1}$). The numbers displayed in the plot are in pb."
        )
    else:
        table.description = (
            "The 95% CL upper limits on the products $\\sigma B_\\mathrm{C}$ and"
            " $\\sigma B_\\mathrm{NC}$, obtained using the cut-based event categorization,"
            " as a function of the scalar masses $m_{\\phi_1}$ and $m_{\\phi_2}$."
            " For the $(m_{\\phi_1}, m_{\\phi_2})$ mass hypotheses (15, 30), (20, 40),"
            " and (30, 60) GeV, only the non-cascade limits are shown. For all other mass"
            " hypotheses, either cascade or non-cascade limits are presented, depending on"
            " whether the cascade decay is kinematically allowed"
            " ($m_{\\phi_2} \\geq 2 m_{\\phi_1}$). The numbers displayed in the plot are in pb."
        )

    table.location = "Results"
    table.keywords["observables"] = ["SIG"]
    table.keywords["reactions"] = [
        "H -> phi_1 phi_2 -> 2 tau 4 b",
        "H -> phi_1 phi_2 -> 2 tau 2 b"
    ]
    #do I need phrases and "particles"?
    data = np.loadtxt(f"NPS25003_inputs/{fileName}", skiprows=0)

    # Column meaning
    y_vals = data[:, 0]   # FIRST column = y bin centers
    x_vals = data[:, 1]   # SECOND column = x bin centers
    z_vals = data[:, 2]   # bin content

    # Build bin edges from centers
    def make_edges(centers):
        centers = np.unique(centers.astype(float))
        edges = np.zeros(len(centers) + 1)
        edges[1:-1] = 0.5 * (centers[1:] + centers[:-1])
        edges[0] = centers[0] - (edges[1] - centers[0])
        edges[-1] = centers[-1] + (centers[-1] - edges[-2])
        return centers, edges

    y_centers, y_edges = make_edges(y_vals)
    x_centers, x_edges = make_edges(x_vals)

    # Independent variables
    phi2_mass = Variable(
        "phi_2 mass",
        is_independent=True,
        is_binned=True,
        units="GeV"
    )

    phi1_mass = Variable(
        "phi_1 mass",
        is_independent=True,
        is_binned=True,
        units="GeV"
    )

    # Map center -> edge tuple
    y_edges_map = {y: (y_edges[i], y_edges[i+1]) for i, y in enumerate(y_centers)}
    x_edges_map = {x: (x_edges[i], x_edges[i+1]) for i, x in enumerate(x_centers)}

    # Only include bins that exist in your data
    phi2_mass.values = [y_edges_map[y] for y in y_vals]
    phi1_mass.values = [x_edges_map[x] for x in x_vals]

    # Dependent variable
    median_limit = Variable(
        "Median limit",
        is_independent=False,
        is_binned=False,
        units="pb"
    )

    median_limit.values = [float(v) for y,x,v in data] 
    median_limit.add_qualifier("SQRT(S)", "13", "TeV")

    # Add to table
    table.add_variable(phi1_mass)
    table.add_variable(phi2_mass)
    table.add_variable(median_limit)

    table.add_image(f"NPS25003_inputs/{imageName}")
    table.add_additional_resource("Original data file", f"NPS25003_inputs/{fileName}", copy_file=True)
    print(table.name)
    return table

In [27]:
def make1DLimitTable_v13(tableName, m1m2_pairs, subfolder, decay, isBDT, isCascade, imageName, description=None):
    """
    Creates a single combined 1D limit Table for one (method × topology) combination.
    
    Files are per-m1:  higgsCombine_a1a2_{decay}_allchannels_allyears_m1_{m1}_limits.txt
    Each file has columns: mh, obs, exp_m2s, exp_m1s, exp_med, exp_p1s, exp_p2s
    where mh is m2. We extract the row matching m2 from the appropriate m1 file.

    Parameters
    ----------
    tableName   : str
    m1m2_pairs  : list of (m1, m2) tuples defining the x-axis points
    subfolder   : str  e.g. "cutbased_root" or "bdtbased_root"
    decay       : str  "4b2t" or "2b2t"
    isBDT       : bool
    isCascade   : bool
    imageName   : str  path to summary plot (relative to NPS25003_inputs/)
    description : str or None  override table description; falls back to isCascade default
    """
    XS_H = 52.38  # ggF H production cross section in pb

    table = Table(tableName)
    table.description = description if description is not None else (
        "B(H -> phi_1 phi_2 -> 2 tau 4b) (%)" if isCascade else "B(H -> phi_1 phi_2 -> 2 tau 2b) (%)"
    )
    table.location = "Results"
    table.keywords["reactions"] = [
        "H -> phi_1 phi_2 -> 2 tau 4 b",
        "H -> phi_1 phi_2 -> 2 tau 2 b"
    ]

    # Cache loaded files so we don't re-read the same m1 file multiple times
    file_cache = {}

    rows = []
    for m1, m2 in m1m2_pairs:
        if m1 not in file_cache:
            fpath = f"NPS25003_inputs/{subfolder}/higgsCombine_a1a2_{decay}_allchannels_allyears_m1_{m1}_limits.txt"
            file_cache[m1] = np.loadtxt(fpath, skiprows=1)
        data = file_cache[m1]
        if data.ndim == 1:
            data = data[np.newaxis, :]  # single-row file edge case
        # mh column (col 0) is m2 — find the matching row
        match = data[np.isclose(data[:, 0], m2)]
        if len(match) == 0:
            raise ValueError(f"m2={m2} not found in m1={m1} file for {decay}/{subfolder}")
        row = match[0]
        obs, exp_m2s, exp_m1s, exp_med, exp_p1s, exp_p2s = (
            row[1], row[2], row[3], row[4], row[5], row[6]
        )
        rows.append((m1, m2, obs, exp_m2s, exp_m1s, exp_med, exp_p1s, exp_p2s))

    # Independent variable: (m1, m2) mass pair as a string label
    mass_pair_var = Variable(
        "(phi_1 mass, phi_2 mass)",
        is_independent=True,
        is_binned=False,
        units="GeV"
    )
    mass_pair_var.values = [f"({m1}, {m2})" for m1, m2, *_ in rows]

    var_observed = Variable("Observed limit",             is_independent=False, is_binned=False, units="pb")
    var_exp_med  = Variable("Expected limit (median)",    is_independent=False, is_binned=False, units="pb")
    var_exp_m1s  = Variable("Expected limit (-1 sigma)",  is_independent=False, is_binned=False, units="pb")
    var_exp_p1s  = Variable("Expected limit (+1 sigma)",  is_independent=False, is_binned=False, units="pb")
    var_exp_m2s  = Variable("Expected limit (-2 sigma)",  is_independent=False, is_binned=False, units="pb")
    var_exp_p2s  = Variable("Expected limit (+2 sigma)",  is_independent=False, is_binned=False, units="pb")

    for v in (var_observed, var_exp_med, var_exp_m1s, var_exp_p1s, var_exp_m2s, var_exp_p2s):
        v.add_qualifier("SQRT(S)", "13", "TeV")

    var_observed.values = [float(r[2]) * XS_H for r in rows]
    var_exp_m2s.values  = [float(r[3]) * XS_H for r in rows]
    var_exp_m1s.values  = [float(r[4]) * XS_H for r in rows]
    var_exp_med.values  = [float(r[5]) * XS_H for r in rows]
    var_exp_p1s.values  = [float(r[6]) * XS_H for r in rows]
    var_exp_p2s.values  = [float(r[7]) * XS_H for r in rows]

    table.add_variable(mass_pair_var)
    table.add_variable(var_observed)
    table.add_variable(var_exp_med)
    table.add_variable(var_exp_m1s)
    table.add_variable(var_exp_p1s)
    table.add_variable(var_exp_m2s)
    table.add_variable(var_exp_p2s)

    table.add_image(f"NPS25003_inputs/NPS-25-003/{imageName}")

    print(table.name)
    return table

In [28]:
import ROOT
ROOT.gROOT.SetBatch(True)
from hepdata_lib import Uncertainty

def makePrefitDistTable(tableName, varSuffix, xLabel, xUnits, imageName, signal_samples, description, channel="mutau"):
    """
    Creates a HEPData Table for one pre-fit distribution subplot,
    summing inclusive histograms across all Run 2 years.

    Parameters
    ----------
    tableName      : str  e.g. "Fig_002-a"
    varSuffix      : str  ROOT histogram key suffix, e.g. "D_zeta"
    xLabel         : str  x-axis label for HEPData
    xUnits         : str  x-axis units, e.g. "GeV" or ""
    imageName      : str  filename under NPS25003_inputs/NPS-25-003/
    signal_samples : list of (proc_name, label)
                     e.g. [("ggh4b2t-70-15", "ggF cascade (m_phi1, m_phi2) = (15,70) GeV"), ...]
    description    : str  table description
    channel        : str  "mutau", "etau", or "emu"
    """
    ROOT_BASE = "/eos/cms/store/group/phys_susy/AN-24-166/aquinn/for_datacards"
    YEAR_FILES = {
        "mutau": [
            f"{ROOT_BASE}/2025-10-13-22h23m_2018-updated_FRsyst-BDT-mt_mtt/out_mutau.root",
            f"{ROOT_BASE}/2025-10-13-22h59m_2017-updated_FRsyst-BDT-mt_mtt/out_mutau.root",
            f"{ROOT_BASE}/2025-10-13-23h11m_2016postVFP-updated_FRsyst-BDT-mt_mtt/out_mutau.root",
            f"{ROOT_BASE}/2025-10-13-23h32m_2016preVFP-updated_FRsyst-BDT-mt_mtt/out_mutau.root",
        ],
        "etau": [
            f"{ROOT_BASE}/2025-10-14-01h37m_2018-updated_FRsyst-BDT-et_mtt/out_etau.root",
            f"{ROOT_BASE}/2025-10-14-01h28m_2017-updated_FRsyst-BDT-et_mtt/out_etau.root",
            f"{ROOT_BASE}/2025-10-14-00h57m_2016postVFP-updated_FRsyst-BDT-et_mtt/out_etau.root",
            f"{ROOT_BASE}/2025-10-14-00h38m_2016preVFP-updated_FRsyst-BDT-et_mtt/out_etau.root",
        ],
        "emu": [
            f"{ROOT_BASE}/2025-10-14-03h34m_2018-updated_FRsyst-BDT-em_mtt/out_emu.root",
            f"{ROOT_BASE}/2025-10-14-04h01m_2017-updated_FRsyst-BDT-em_mtt/out_emu.root",
            f"{ROOT_BASE}/2025-10-14-04h52m_2016postVFP-updated_FRsyst-BDT-em_mtt/out_emu.root",
            f"{ROOT_BASE}/2025-10-14-05h28m_2016preVFP-updated_FRsyst-BDT-em_mtt/out_emu.root",
        ],
    }[channel]

    SM_HIGGS_PROCS = [
        "ggh_htt", "ggh_hww", "qqh_htt", "qqh_hww",
        "Zh_htt", "Zh_hww", "Wh_htt", "Wh_hww", "tth",
    ]

    if channel == "emu":
        INDIV_BG_PROCS = ["embedded", "qcd", "WJ", "ZJ", "ttbar", "ST", "VV"]
    else:
        INDIV_BG_PROCS = ["embedded", "fake", "ZJ", "ttbar", "ST", "VV"]

    channel_label = {"mutau": "mu tau", "etau": "e tau", "emu": "e mu"}[channel]

    def _get(directory, key):
        h = directory.Get(key)
        if not h:
            return None
        h = h.Clone()
        h.SetDirectory(0)
        return h

    def _add(hists, key, h):
        if h is None:
            return
        if key not in hists:
            hists[key] = h
        else:
            hists[key].Add(h)

    hists = {}

    for fpath in YEAR_FILES:
        f = ROOT.TFile.Open(fpath)
        if not f or f.IsZombie():
            print(f"WARNING: could not open {fpath}")
            continue
        inc = f.Get("inclusive")

        _add(hists, "data_obs", _get(inc, f"data_obs_{varSuffix}"))

        for proc in INDIV_BG_PROCS:
            _add(hists, proc, _get(inc, f"{proc}_{varSuffix}"))

        sm_total = None
        for p in SM_HIGGS_PROCS:
            h = _get(inc, f"{p}_{varSuffix}")
            if h is not None:
                if sm_total is None:
                    sm_total = h
                else:
                    sm_total.Add(h)
        _add(hists, "SM_Higgs", sm_total)

        for proc_name, _ in signal_samples:
            _add(hists, proc_name, _get(inc, f"{proc_name}_{varSuffix}"))

        f.Close()

    def _sum_hists(keys):
        total = None
        for k in keys:
            h = hists.get(k)
            if h is None:
                continue
            if total is None:
                total = h.Clone()
            else:
                total.Add(h)
        return total

    # "Other" for mutau/etau = Z/gamma*->ee/mumu (ZJ), single top (ST), diboson (VV), SM Higgs
    # "Other" for emu = W+jets (WJ), ZJ, ST, VV, SM Higgs
    if channel == "emu":
        grouped_bkg = [
            (r"$Z\to\tau\tau$",  _sum_hists(["embedded"])),
            ("QCD",              _sum_hists(["qcd"])),
            (r"$t\bar{t}$+jets", _sum_hists(["ttbar"])),
            ("Other",            _sum_hists(["WJ", "ZJ", "ST", "VV", "SM_Higgs"])),
        ]
    else:
        grouped_bkg = [
            (r"$Z\to\tau\tau$",  _sum_hists(["embedded"])),
            (r"Jet$\to\tau_h$",  _sum_hists(["fake"])),
            (r"$t\bar{t}$+jets", _sum_hists(["ttbar"])),
            ("Other",            _sum_hists(["ZJ", "ST", "VV", "SM_Higgs"])),
        ]

    hists["total_bkg"] = _sum_hists(INDIV_BG_PROCS + ["SM_Higgs"])

    table = Table(tableName)
    table.description = description
    table.location = "Results"
    table.keywords["reactions"] = [
        "H -> phi_1 phi_2 -> 2 tau 4 b",
        "H -> phi_1 phi_2 -> 2 tau 2 b",
    ]

    href = hists["data_obs"]
    nbins = href.GetNbinsX()
    bin_edges = [
        (href.GetXaxis().GetBinLowEdge(i), href.GetXaxis().GetBinUpEdge(i))
        for i in range(1, nbins + 1)
    ]

    x_var = Variable(xLabel, is_independent=True, is_binned=True, units=xUnits)
    x_var.values = bin_edges
    table.add_variable(x_var)

    def _dep_var(label, hist, vtype, with_unc=True):
        v = Variable(label, is_independent=False, is_binned=False, units="Events")
        v.add_qualifier("SQRT(S)", "13", "TeV")
        v.add_qualifier("channel", channel_label)
        v.add_qualifier("type", vtype)
        vals = [hist.GetBinContent(i) for i in range(1, nbins + 1)]
        v.values = vals
        if with_unc:
            unc = Uncertainty("stat", is_symmetric=True)
            unc.values = [hist.GetBinError(i) for i in range(1, nbins + 1)]
            v.add_uncertainty(unc)
        return v

    table.add_variable(_dep_var("Data",             hists["data_obs"],  "data"))
    table.add_variable(_dep_var("Total background", hists["total_bkg"], "total background"))
    for label, hist in grouped_bkg:
        if hist is not None:
            table.add_variable(_dep_var(label, hist, "background"))
    for proc_name, label in signal_samples:
        if hists.get(proc_name):
            table.add_variable(_dep_var(label, hists[proc_name], "signal", with_unc=False))

    table.add_image(f"NPS25003_inputs/NPS-25-003/{imageName}")
    print(table.name)
    return table

## Main Function

The `Submission` object represents the whole HEPData entry and thus carries the top-level meta data that is equally valid for all the tables and variables you may want to enter. The object is also used to create the physical submission files you will upload to the HEPData web interface.

When using `hepdata_lib` to make an entry, you always need to create a `Submission` object. 

In [29]:
def main():
    submission = Submission()
    submission.read_abstract("NPS25003_inputs/abstract.txt")

    #ADL
    submission.add_additional_resource("ADL file", "NPS25003_inputs/NPS25003.adl", copy_file=True)

    #Production cross section
    submission.add_link("Standard ggF and VBF cross-sections from Handbook of LHC Cross-sections", "http://arxiv.org/abs/arXiv:1610.07922") 

    #Signal model UFO Files
    submission.add_link("Signal Model UFO files", "https://gitlab.com/apapaefs/twosinglet")

    #Generator Process cards
    submission.add_link("Generator Process Cards", "https://github.com/cms-sw/genproductions/pull/2705") 

    #Small set of input vectors & ML outputs
    submission.add_link("BDT Models", "https://github.com/Aaravind96/aabbttBDT/tree/preservation/BDTmodels")

    #Statistical model
    submission.add_link("Datacards", "https://gitlab.cern.ch/cms-analysis/nps/nps-25-003/datacards/-/tree/master/input?ref_type=heads") 
    
    #Final wiki
    submission.add_link("Wiki", "https://cms-results.web.cern.ch/cms-results/public-results/publications/NPS-25-003/") 

    ##################
    # 2D Limit plots #
    ##################
    version_folder = "v11_limits"
    plots = "NPS-25-003"

    table_configs_2D = [
        # (name,                               isBDT, subfolder,    channel,         imageName)
        ("Fig_009-d_2D_BDT_allchannels",       True,  "bdt_based",   "allchannels", f"{plots}/Figure_009-d.pdf"),
        ("Fig_009-c_2D_BDT_emu",               True,  "bdt_based",   "emu",         f"{plots}/Figure_009-c.pdf"),
        ("Fig_009-a_2D_BDT_mutau",             True,  "bdt_based",   "mutau",       f"{plots}/Figure_009-a.pdf"),
        ("Fig_009-b_2D_BDT_etau",              True,  "bdt_based",   "etau",        f"{plots}/Figure_009-b.pdf"),
        ("Fig_015-d_2D_cutbased_allchannels",  False, "cut_based",   "allchannels", f"{plots}/Figure_015-d.pdf"),
        ("Fig_015-c_2D_cutbased_emu",          False, "cut_based",   "emu",         f"{plots}/Figure_015-c.pdf"),
        ("Fig_015-a_2D_cutbased_mutau",        False, "cut_based",   "mutau",       f"{plots}/Figure_015-a.pdf"),
        ("Fig_015-b_2D_cutbased_etau",         False, "cut_based",   "etau",        f"{plots}/Figure_015-b.pdf"),
    ]

    for name, isBDT, subfolder, channel, imageName in table_configs_2D:
        submission.add_table(make2DLimitTable(
            name,
            isBDT,
            f"{version_folder}/{subfolder}/median_limits_{channel}.txt",
            imageName 
        ))

    ######################
    # 1D Limit plots v13 #
    ######################

    _noncascade_pairs = [
        (15,20),(15,30),(20,30),(20,40),(30,40),(30,50),(30,60),
        (40,50),(40,60),(40,70),(40,80),(50,60),(50,70),
    ]
    _cascade_pairs = [
        (15,30),(15,40),(15,50),(15,60),(15,70),(15,80),(15,90),(15,100),(15,110),
        (20,40),(20,50),(20,60),(20,70),(20,80),(20,90),(20,100),
        (30,60),(30,70),(30,80),(30,90),
    ]

    descriptions_1D = {
        "Fig_013": (
            r"The observed (points) and median expected (dotted line) 95% CL upper limits on"
            r" $\sigma B_\mathrm{C}$ for the cascade scenario using the cut-based event"
            r" categorization and the fit to the $m_{\tau\tau}$ distribution, for different"
            r" mass hypotheses $(m_{\phi_1}, m_{\phi_2})$. The horizontal bars on the points"
            r" are for better legibility only. The green and yellow regions show the 68 and 95%"
            r" expected range for the median value, respectively."
        ),
        "Fig_007": (
            r"The observed (points) and median expected (dotted line) 95% CL upper limits on"
            r" $\sigma B_\mathrm{C}$ for the cascade scenario using the BDT-based event"
            r" categorization and the fit to the $m_{\tau\tau}$ distribution, for different"
            r" mass hypotheses $(m_{\phi_1}, m_{\phi_2})$. The horizontal bars on the points"
            r" are for better legibility only. The green and yellow regions show the 68 and 95%"
            r" expected range for the median value, respectively."
        ),
        "Fig_014": (
            r"The observed (points) and median expected (dotted line) 95% CL upper limits on"
            r" $\sigma B_\mathrm{NC}$ for the non-cascade scenario using the cut-based event"
            r" categorization and the fit to the $m_{\tau\tau}$ distribution, for different"
            r" mass hypotheses $(m_{\phi_1}, m_{\phi_2})$. The horizontal bars on the points"
            r" are for better legibility only. The green and yellow regions show the 68 and 95%"
            r" expected range for the median value, respectively."
        ),
        "Fig_008": (
            r"The observed (points) and median expected (dotted line) 95% CL upper limits on"
            r" $\sigma B_\mathrm{NC}$ for the non-cascade scenario using the BDT-based event"
            r" categorization and the fit to the $m_{\tau\tau}$ distribution, for different"
            r" mass hypotheses $(m_{\phi_1}, m_{\phi_2})$. The horizontal bars on the points"
            r" are for better legibility only. The green and yellow regions show the 68 and 95%"
            r" expected range for the median value, respectively."
        ),
    }

    table_configs_1D_v13 = [
        # (tableName,   subfolder,       decay,  isBDT, isCascade, pairs,         plot_tag,   imageName)
        ("Fig_013", "cutbased_root", "4b2t", False, True,  _cascade_pairs,    "cutbased", "Figure_013.pdf"),
        ("Fig_007", "bdtbased_root", "4b2t", True,  True,  _cascade_pairs,    "bdt_",     "Figure_007.pdf"),
        ("Fig_014", "cutbased_root", "2b2t", False, False, _noncascade_pairs, "cutbased", "Figure_014.pdf"),
        ("Fig_008", "bdtbased_root", "2b2t", True,  False, _noncascade_pairs, "bdt_",     "Figure_008.pdf"),
    ]

    for tableName, subfolder, decay, isBDT, isCascade, pairs, plot_tag, imageName in table_configs_1D_v13:
        submission.add_table(
            make1DLimitTable_v13(tableName, pairs, subfolder, decay, isBDT, isCascade, imageName,
                                 description=descriptions_1D.get(tableName) or None)
        )

    ##############################
    # Pre-fit distributions      #
    # Figure 002 (mutau channel) #
    ##############################

    # Process name format: {mode}{decay}-{m2}-{m1}
    #   mode:  "ggh" (ggF) or "vbf"
    #   decay: "4b2t" (cascade) or "2b2t" (non-cascade)
    #   e.g. "ggh4b2t-70-15" = ggF cascade, m_phi2=70, m_phi1=15 GeV
    _prefit_signals = [
        ("ggh4b2t-70-15", r"ggF cascade $(m_{\phi_1},m_{\phi_2})=(15,70)$ GeV"),
        ("vbf4b2t-70-15", r"VBF cascade $(m_{\phi_1},m_{\phi_2})=(15,70)$ GeV"),
        ("ggh2b2t-30-20", r"ggF non-cascade $(m_{\phi_1},m_{\phi_2})=(20,30)$ GeV"),
        ("vbf2b2t-30-20", r"VBF non-cascade $(m_{\phi_1},m_{\phi_2})=(20,30)$ GeV"),
        ("ggh4b2t-80-30", r"ggF cascade $(m_{\phi_1},m_{\phi_2})=(30,80)$ GeV"),
        ("vbf4b2t-80-30", r"VBF cascade $(m_{\phi_1},m_{\phi_2})=(30,80)$ GeV"),
        ("ggh2b2t-60-40", r"ggF non-cascade $(m_{\phi_1},m_{\phi_2})=(40,60)$ GeV"),
        ("vbf2b2t-60-40", r"VBF non-cascade $(m_{\phi_1},m_{\phi_2})=(40,60)$ GeV"),
    ]

    _prefit_desc = (
        r"Pre-fit distributions of {var} including underflow and overflow bins,"
        r" for preselected events with at least one b-tagged jet for the $\mu\tau_h$ channel,"
        r" without any SR requirements. Data are shown by the markers with vertical bars"
        r" and backgrounds by the colored histograms. The hatched areas show the combination"
        r" of statistical and shape systematic uncertainties. The colored open histograms display"
        r" predicted signal distributions for two cascade and two non-cascade decays."
        r" The lower panel shows the ratio of data to the sum of the predicted background events."
    )

    table_configs_prefit = [
        # (tableName,  varSuffix,        xLabel,                                                xUnits, imageName)
        ("Fig_002-a", "D_zeta",        r"$D_\zeta$",                                          "GeV",  "Figure_002-a.pdf"),
        ("Fig_002-b", "m_btautau_vis", r"$m^\mathrm{vis}(\tau\tau b_1)$",                     "GeV",  "Figure_002-b.pdf"),
        ("Fig_002-c", "mtMET_1",       r"$m_\mathrm{T}(\mu, p_\mathrm{T}^\mathrm{miss})$",    "GeV",  "Figure_002-c.pdf"),
        ("Fig_002-d", "mtMET_2",       r"$m_\mathrm{T}(\tau_h, p_\mathrm{T}^\mathrm{miss})$", "GeV",  "Figure_002-d.pdf"),
    ]

    for tableName, varSuffix, xLabel, xUnits, imageName in table_configs_prefit:
        submission.add_table(makePrefitDistTable(
            tableName, varSuffix, xLabel, xUnits, imageName,
            _prefit_signals,
            _prefit_desc.format(var=xLabel),
        ))

    ######################################
    # Pre-fit BDT score distributions    #
    # Figure 003 (mutau, etau, emu)      #
    ######################################

    _fig3_desc = (
        "Pre-fit BDT score distribution for preselected events with at least one"
        " b-tagged jet for the {channel_label} channel, without any SR requirements."
        " Data are shown by the markers with vertical bars and various backgrounds by"
        " the colored histograms. The combination of statistical and shape systematic"
        " uncertainties is displayed with the hatched areas. The colored open histograms"
        " display the predicted signal distribution for two cascade decays and two"
        " non-cascade decays, with four different values of $m_{{\\phi_1}}$ and"
        " $m_{{\\phi_2}}$ masses, for an assumed branching fraction of 100%."
        " The lower panel shows the ratio of the data to the sum of the predicted"
        " number of background events. The vertical bars on the points show the"
        " statistical uncertainty in the ratio."
    )

    _channel_labels = {
        "mutau": r"$\mu\tau_h$",
        "etau":  r"$e\tau_h$",
        "emu":   r"$e\mu$",
    }

    table_configs_fig3 = [
        # (tableName,  channel,  imageName)
        ("Fig_003-a", "mutau", "Figure_003-a.pdf"),
        ("Fig_003-b", "etau",  "Figure_003-b.pdf"),
        ("Fig_003-c", "emu",   "Figure_003-c.pdf"),
    ]

    for tableName, channel, imageName in table_configs_fig3:
        submission.add_table(makePrefitDistTable(
            tableName, "bdtscore", "BDT score", "", imageName,
            _prefit_signals,
            _fig3_desc.format(channel_label=_channel_labels[channel]),
            channel=channel,
        ))

    ##########
    # Cutflow#
    ##########
    table = Table("Cutflow")
    table.description = (
        "Cutflow showing event yields after each selection step for two "
        "signal mass points, labelled by (m1, m2) in GeV."
    )
    table.location = "Auxiliary material"
    table.keywords["observables"] = ["N"]

    cuts = Variable("Selection step", is_independent=True, is_binned=False, units="")
    cuts.values = [
        "No cuts applied",
        "After event pre-selections",
        "SR1_1b",
        "SR2_1b",
        "SR3_1b",
        "SR4_1b",
        "SR1_2b",
        "SR2_2b",
    ]
    table.add_variable(cuts)

    # Signal point (60, 40)
    sig_60_40 = Variable("Yield", is_independent=False, is_binned=False, units="")
    sig_60_40.values = [
        326685418,
        2146.033512,
        593.1758018,
        490.4246293,
        338.9056403,
        188.6788008,
        242.2365516,
        173.7265271,
    ]
    sig_60_40.add_qualifier("mass point (m1, m2)", "(60, 40) GeV")
    sig_60_40.add_qualifier("SQRT(S)", 13, "TeV")
    table.add_variable(sig_60_40)

    # Signal point (80, 30)
    sig_80_30 = Variable("Yield", is_independent=False, is_binned=False, units="")
    sig_80_30.values = [
        2780469921,
        75.53763655,
        29.13513019,
        15.4912386,
        7.48410232,
        3.17092978,
        9.127757683,
        7.005560957,
    ]
    sig_80_30.add_qualifier("mass point (m1, m2)", "(80, 30) GeV")
    sig_80_30.add_qualifier("SQRT(S)", 13, "TeV")
    table.add_variable(sig_80_30)
    submission.add_table(table) 

    for t in submission.tables:
        table.keywords["cmenergies"] = [13000]
    outdir = "NPS25003_output"
    print("Tables:", [t.name for t in submission.tables])
    
    submission.create_files(outdir, remove_old=True)

In [30]:
if __name__ == "__main__":
    main()

Fig_009-d_2D_BDT_allchannels
Fig_009-c_2D_BDT_emu
Fig_009-a_2D_BDT_mutau
Fig_009-b_2D_BDT_etau
Fig_015-d_2D_cutbased_allchannels
Fig_015-c_2D_cutbased_emu
Fig_015-a_2D_cutbased_mutau
Fig_015-b_2D_cutbased_etau
Fig_013
Fig_007
Fig_014
Fig_008
Fig_002-a
Fig_002-b
Fig_002-c
Fig_002-d
Fig_003-a
Fig_003-b
Fig_003-c
Tables: ['Fig_009-d_2D_BDT_allchannels', 'Fig_009-c_2D_BDT_emu', 'Fig_009-a_2D_BDT_mutau', 'Fig_009-b_2D_BDT_etau', 'Fig_015-d_2D_cutbased_allchannels', 'Fig_015-c_2D_cutbased_emu', 'Fig_015-a_2D_cutbased_mutau', 'Fig_015-b_2D_cutbased_etau', 'Fig_013', 'Fig_007', 'Fig_014', 'Fig_008', 'Fig_002-a', 'Fig_002-b', 'Fig_002-c', 'Fig_002-d', 'Fig_003-a', 'Fig_003-b', 'Fig_003-c', 'Cutflow']
Note that bins with zero content should preferably be omitted completely from the HEPData table.
Note that bins with zero content should preferably be omitted completely from the HEPData table.
Note that bins with zero content should preferably be omitted completely from the HEPData table.
Note tha

In [31]:
!cat NPS25003_output/submission.yaml

---
additional_resources:
- description: Created with hepdata_lib 0.20.0
  location: https://doi.org/10.5281/zenodo.1217998
- description: ADL file
  location: NPS25003.adl
- description: Standard ggF and VBF cross-sections from Handbook of LHC Cross-sections
  location: http://arxiv.org/abs/arXiv:1610.07922
- description: Signal Model UFO files
  location: https://gitlab.com/apapaefs/twosinglet
- description: Generator Process Cards
  location: https://github.com/cms-sw/genproductions/pull/2705
- description: BDT Models
  location: https://github.com/Aaravind96/aabbttBDT/tree/preservation/BDTmodels
- description: Datacards
  location: https://gitlab.cern.ch/cms-analysis/nps/nps-25-003/datacards/-/tree/master/input?ref_type=heads
- description: Wiki
  location: https://cms-results.web.cern.ch/cms-results/public-results/publications/NPS-25-003/
comment: A search for Higgs boson decays to a pair of neutral scalars phi_1 and phi_2
  with unequal masses is performed in final states with b 

In [32]:
!ls NPS25003_output

Figure_002-a.png	     fig_009-c_2d_bdt_emu.yaml
Figure_002-b.png	     fig_009-d_2d_bdt_allchannels.yaml
Figure_002-c.png	     fig_013.yaml
Figure_002-d.png	     fig_014.yaml
Figure_003-a.png	     fig_015-a_2d_cutbased_mutau.yaml
Figure_003-b.png	     fig_015-b_2d_cutbased_etau.yaml
Figure_003-c.png	     fig_015-c_2d_cutbased_emu.yaml
Figure_007.png		     fig_015-d_2d_cutbased_allchannels.yaml
Figure_008.png		     median_limits_allchannels.txt
Figure_009-a.png	     median_limits_emu.txt
Figure_009-b.png	     median_limits_etau.txt
Figure_009-c.png	     median_limits_mutau.txt
Figure_009-d.png	     submission.yaml
Figure_013.png		     thumb_Figure_002-a.png
Figure_014.png		     thumb_Figure_002-b.png
Figure_015-a.png	     thumb_Figure_002-c.png
Figure_015-b.png	     thumb_Figure_002-d.png
Figure_015-c.png	     thumb_Figure_003-a.png
Figure_015-d.png	     thumb_Figure_003-b.png
NPS25003.adl		     thumb_Figure_003-c.png
cutflow.yaml		     thumb_Figure_007.png
fig_002-a.yaml		     thumb_Fig

In [33]:
!ls submission.tar.gz

submission.tar.gz
